In [6]:
from pathlib import Path
import subprocess
import sys

# Use explicit paths first (works in local Jupyter or mounted environments).
score_script = Path("/Users/amir/Documents/New project/src/score.py")
transcripts_dir = Path("/Users/amir/Documents/New project/data/transcripts")


def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()

    # 1) Check cwd and its parents first.
    for base in [cwd, *cwd.parents]:
        if (base / "src" / "score.py").exists() and (base / "data" / "transcripts").exists():
            return base

    # 2) Check common Colab/project locations.
    candidates = [
        Path("/content/New project"),
        Path("/content/new-project"),
        Path("/content/project"),
        Path("/content/drive/MyDrive/New project"),
        Path("/content/drive/MyDrive/project"),
    ]
    for base in candidates:
        if (base / "src" / "score.py").exists() and (base / "data" / "transcripts").exists():
            return base

    # 3) Last resort: shallow search under /content.
    content_root = Path("/content")
    if content_root.exists():
        for candidate in content_root.rglob("score.py"):
            if candidate.parent.name != "src":
                continue
            base = candidate.parent.parent
            if (base / "data" / "transcripts").exists():
                return base

    raise FileNotFoundError(
        "Could not find project root containing both src/score.py and data/transcripts."
    )


if score_script.exists() and transcripts_dir.exists():
    repo_root = score_script.parent.parent
else:
    repo_root = find_repo_root()
    score_script = repo_root / "src" / "score.py"
    transcripts_dir = repo_root / "data" / "transcripts"

print(f"Using repo root: {repo_root}")
print(f"Running scorer on transcripts in: {transcripts_dir}")
subprocess.run([sys.executable, str(score_script)], check=True, cwd=str(repo_root))

FileNotFoundError: Could not find project root containing both src/score.py and data/transcripts.